# MS4215 — Lab 2: Espresso Extraction
## R first, then Python / Google Colab

This notebook teaches the same statistics as the lecturer's **R** lab and then reproduces them in **Python**.

### Learning goals
1. Construct the design matrix \(X\).
2. Compute \((X^TX)^{-1}\).
3. Calculate \(\hat\beta=(X^TX)^{-1}X^Ty\).
4. Verify with a regression package.
5. Interpret the overall F-test and individual t-tests.
6. Check multicollinearity using VIF.

### R → Python
| Task | R | Python |
|---|---|---|
| Read data | `read_csv()` | `pd.read_csv()` |
| Transpose | `t(X)` | `X.T` |
| Matrix multiply | `%*%` | `@` |
| Inverse | `solve()` | `np.linalg.inv()` |
| Regression | `lm()` | `statsmodels.OLS()` |
| VIF | `car::vif()` | `variance_inflation_factor()` |

The syntax changes. **The mathematics does not.**


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor
pd.set_option("display.precision", 4)


## Load the dataset from the web

No local C: or E: path is needed. Both R and Python can read the same public CSV.


In [ ]:
DATA_URL = "https://erikdownes.github.io/resources/ms4215/data/espresso.csv"
espresso = pd.read_csv(DATA_URL)
espresso


## Inspect before modelling

Always check shape, column names, data types, missing values and ranges before fitting a model.


In [ ]:
print("Shape:", espresso.shape)
print("\nTypes:")
print(espresso.dtypes)
print("\nMissing:")
print(espresso.isna().sum())
espresso.describe()


## (i) Construct the design matrix

For multiple linear regression,

\[
y=X\beta+\varepsilon
\]

The leading column of ones estimates the intercept.


In [ ]:
y = espresso["extraction_yield"].to_numpy()

X = np.column_stack([
    np.ones(len(espresso)),
    espresso["grind_size"],
    espresso["water_temp"],
    espresso["shot_time"]
])

print("X shape:", X.shape)
print(X)


### R equivalent

```r
y <- espresso$extraction_yield
X <- cbind(1, espresso$grind_size, espresso$water_temp, espresso$shot_time)
X
```


## (ii) Compute \((X^TX)^{-1}\)


In [ ]:
XtX = X.T @ X
XtX_inv = np.linalg.inv(XtX)
print("X^T X:")
print(XtX)
print("\nInverse:")
print(XtX_inv)


### R equivalent

```r
XtX <- t(X) %*% X
XtX_inv <- solve(XtX)
```


## (iii) Calculate the coefficients manually

\[
\hat\beta=(X^TX)^{-1}X^Ty
\]


In [ ]:
beta_hat = XtX_inv @ X.T @ y
names = ["Intercept", "grind_size", "water_temp", "shot_time"]
pd.Series(beta_hat, index=names, name="coefficient")


The fitted equation is approximately:

\[
\widehat{yield}=13.864-0.0361(grind\ size)+0.2696(water\ temp)+0.0385(shot\ time)
\]

Interpret a multiple-regression coefficient **holding the other included predictors constant**.


## (iv) Verify with a regression package


In [ ]:
X_sm = sm.add_constant(espresso[["grind_size", "water_temp", "shot_time"]])
model = sm.OLS(espresso["extraction_yield"], X_sm).fit()
print(model.summary())


In [ ]:
pd.DataFrame({
    "manual_matrix": beta_hat,
    "statsmodels": model.params.to_numpy(),
    "difference": beta_hat - model.params.to_numpy()
}, index=names)


### R equivalent

```r
fit <- lm(extraction_yield ~ grind_size + water_temp + shot_time, data = espresso)
summary(fit)
```

The R and Python coefficients should agree because both are fitting the same OLS model.


## (v) Significance

The **overall F-test** asks whether the predictors jointly improve on an intercept-only model.

For these data, the model is strongly significant overall.

The **individual t-tests** ask whether each partial coefficient differs from zero after controlling for the other predictors.

At the 5% level:
- grind size: significant
- water temperature: significant
- shot time: not significant


In [ ]:
results = pd.DataFrame({
    "coefficient": model.params,
    "std_error": model.bse,
    "t": model.tvalues,
    "p_value": model.pvalues
})
results["significant_at_5pct"] = results["p_value"] < 0.05
results


## (vi) VIF and multicollinearity

VIF asks whether a predictor can be strongly explained by the **other predictors**.

Rules of thumb often used:
- about 1: little collinearity
- above 5: investigate
- above 10: often serious

These are guides, not laws.


In [ ]:
predictors = espresso[["grind_size", "water_temp", "shot_time"]]
vif = pd.DataFrame({
    "variable": predictors.columns,
    "VIF": [variance_inflation_factor(predictors.values, i)
            for i in range(predictors.shape[1])]
})
vif


The VIFs are close to 1, so multicollinearity is not the reason shot time is non-significant.


## Visualise the raw relationships


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["grind_size", "water_temp", "shot_time"]):
    ax.scatter(espresso[col], espresso["extraction_yield"])
    ax.set_xlabel(col)
    ax.set_ylabel("extraction_yield")
    ax.set_title(f"Yield vs {col}")
plt.tight_layout()
plt.show()


## Diagnostics and LINE assumptions

OLS inference relies on the model being reasonable.

- **L** — linearity
- **I** — independence
- **N** — approximate normality of residuals for inference
- **E** — equal variance

With only 15 observations, diagnostics need cautious interpretation.


In [ ]:
fitted = model.fittedvalues
resid = model.resid

plt.figure(figsize=(7,5))
plt.scatter(fitted, resid)
plt.axhline(0)
plt.xlabel("Fitted values")
plt.ylabel("Residuals")
plt.title("Residuals vs fitted")
plt.show()

sm.qqplot(resid, line="45")
plt.title("Normal Q-Q plot")
plt.show()


## What Erik should be able to say without code

**Why multiple regression?**  
It estimates the contribution of each predictor while the other included predictors are controlled.

**Why build the design matrix manually?**  
It reveals what `lm()` and `statsmodels` are doing rather than treating the software as magic.

**Why can the model be significant while shot time is not?**  
The F-test evaluates predictors jointly. Each t-test evaluates one coefficient conditional on the others.

**Why check VIF?**  
Highly correlated predictors can make separate coefficients unstable and inflate standard errors.


## Run the lecturer's version in R / Posit Cloud

```r
library(tidyverse)
library(car)

espresso <- read_csv(
  "https://erikdownes.github.io/resources/ms4215/data/espresso.csv"
)

y <- espresso$extraction_yield
X <- cbind(1, espresso$grind_size, espresso$water_temp, espresso$shot_time)

XtX <- t(X) %*% X
XtX_inv <- solve(XtX)
beta_hat <- XtX_inv %*% t(X) %*% y
beta_hat

fit <- lm(
  extraction_yield ~ grind_size + water_temp + shot_time,
  data = espresso
)

summary(fit)
vif(fit)
plot(fit)
```

Both languages read the **same cloud-hosted CSV**.


## Recommended study loop

1. Read the statistical question.
2. Say what concept is being tested.
3. Do the lecturer's R version.
4. Explain the result in ordinary English.
5. Reproduce it in Python.
6. Compare the two outputs.
7. Change an input or model and predict what will happen before running it.

That is the point of the second implementation: **understanding**, not replacing R.
